# SIMULATION EXPERIMENT DESIGN

## Basic:
- 10 members
- x time steps
- 1 time step = 1 day

## Variables:
### Independent
- Leader aperture: NI, LC, LI, HC, HI, F
- Network structure: Random, community, core periphery

### Dependent
- Member emotion convergence (existence, speed)
- Member emotion valence
- Network evolution


## What is being tested
1. How long until convergence is reached?
2. Is convergence speed insensitive to network?
3. Do different leaders lead to: quicker convergence, more stability

#### Notes
- **Convergence** is defined as having an average team emotion with small standard deviation (*suggested sd ≤ 0.01*)
- Should we control interventions more strictly? E.g. intervention every x steps, intervention during first/last x steps

In [1]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load and set up results

In [2]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent
RUNNING_DIR = PROJECT_ROOT / "running"
SRC_DIR = PROJECT_ROOT / "src"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

for path in [str(PROJECT_ROOT), str(RUNNING_DIR), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

In [3]:
# Build tables
def get_condition_key(results):
    return (results.metadata.get("structure"), results.metadata.get("leader_style"))

def build_avg_emotion_df(results_list):
    rows = []
    for results in results_list:
        structure = results.metadata.get("structure")
        leader_style = results.metadata.get("leader_style")
        condition_name = results.metadata.get("condition_name")
        for t, avg_emotion in enumerate(results.avg_emotion_history):
            rows.append({
                "run_id": results.run_id,
                "seed": results.seed,
                "condition_name": condition_name,
                "structure": structure,
                "leader_style": leader_style,
                "time": t,
                "avg_member_emotion": avg_emotion
            })
    return pd.DataFrame(rows)

def build_intervention_frequency_df(results_list):
    rows = []
    for results in results_list:
        structure = results.metadata.get("structure")
        leader_style = results.metadata.get("leader_style")
        condition_name = results.metadata.get("condition_name")

        n_t = len(results.avg_emotion_history)
        intervention_log = (results.intervention_log if results.intervention_log is not None else {})
        for t in range(n_t):
            action = intervention_log.get(t, 0)
            rows.append({
                "run_id": results.run_id,
                "seed": results.seed,
                "condition_name": condition_name,
                "structure": structure,
                "leader_style": leader_style,
                "time": t,
                "action": action, # raw action strength
                "intervened": int(action != 0),   # convenience binary version

                # one-hot indicators
                "action_no": int(action == 0),
                "action_weak": int(action == 1),
                "action_medium": int(action == 2),
                "action_strong": int(action == 3),
            })
    return pd.DataFrame(rows)

def build_intervention_summary_df(results_list):
    rows = []
    for results in results_list:
        intervention_log = (results.intervention_log if results.intervention_log is not None else {})
        intervention_times = sorted(intervention_log.keys())
        actions = list(intervention_log.values())

        rows.append({
            "run_id": results.run_id,
            "seed": results.seed,
            "condition_name": results.metadata.get("condition_name"),
            "structure": results.metadata.get("structure"),
            "leader_style": results.metadata.get("leader_style"),

            # overall intervention info
            "num_interventions": len(intervention_times),
            "first_intervention_time": (intervention_times[0] if intervention_times else np.nan),
            "had_any_intervention": int(len(intervention_times) > 0),

            # strength-specific counts
            "num_weak": sum(a == 1 for a in actions),
            "num_medium": sum(a == 2 for a in actions),
            "num_strong": sum(a == 3 for a in actions),

            # proportions among interventions
            "prop_weak": (np.mean([a == 1 for a in actions]) if actions else np.nan),
            "prop_medium": (np.mean([a == 2 for a in actions]) if actions else np.nan),
            "prop_strong": (np.mean([a == 3 for a in actions]) if actions else np.nan),
        })
    return pd.DataFrame(rows)

def build_intervention_raster_df(results_list):
    rows = []
    for results in results_list:
        intervention_log = (results.intervention_log if results.intervention_log is not None else {})
        for t, action in intervention_log.items():
            rows.append({
                "run_id": results.run_id,
                "seed": results.seed,
                "condition_name": results.metadata.get("condition_name"),
                "structure": results.metadata.get("structure"),
                "leader_style": results.metadata.get("leader_style"),
                "time": t,
                "action": action,
                "action_label": {
                    0: "No",
                    1: "Weak",
                    2: "Medium",
                    3: "Strong",
                }.get(action, str(action))
            })
    return pd.DataFrame(rows)

def build_rl_action_df(results_list):
    rows = []
    for results in results_list:
        if not hasattr(results, "rl_actions"):  continue
        if results.rl_actions is None: continue
        for t, action in enumerate(results.rl_actions):
            rows.append({
                "run_id": results.run_id,
                "seed": results.seed,
                "condition_name": results.metadata.get("condition_name"),
                "structure": results.metadata.get("structure"),
                "leader_style": results.metadata.get("leader_style"),
                "time": t,
                "action": action,
                "intervened": int(action != 0),
            })
    return pd.DataFrame(rows)

def build_rl_quality_df(results_list):
    rows = []
    for results in results_list:
        if not hasattr(results, "rl_quality"): continue
        if results.rl_quality is None:  continue
        for t, quality in enumerate(results.rl_quality):
            rows.append({
                "run_id": results.run_id,
                "seed": results.seed,
                "condition_name": results.metadata.get("condition_name"),
                "structure": results.metadata.get("structure"),
                "leader_style": results.metadata.get("leader_style"),
                "time": t,
                "quality": quality
            })
    return pd.DataFrame(rows)

def build_homophily_df(results_list):
    rows = []
    for results in results_list:
        if not hasattr(results, "homophily_history"): continue
        if results.homophily_history is None: continue
        for t, value in enumerate(results.homophily_history):
            rows.append({
                "run_id": results.run_id,
                "seed": results.seed,
                "condition_name": results.metadata.get("condition_name"),
                "structure": results.metadata.get("structure"),
                "leader_style": results.metadata.get("leader_style"),
                "time": t,
                "homophily": value,
            })
    return pd.DataFrame(rows)

def load_result_pickle(path: str | Path):
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)

def load_all_result_pickles(results_dir: str | Path):
    results_dir = Path(results_dir)
    pickle_paths = sorted(p for p in results_dir.rglob("simulation_result_run_*.pkl") if "_qtable" not in p.name)
    if not pickle_paths:
        raise FileNotFoundError(f"No result pickle files were found under: {results_dir}")
    return [load_result_pickle(path) for path in pickle_paths], pickle_paths

def get_recent_batch_dirs(outputs_dir, n=4):
    outputs_dir = Path(outputs_dir)
    subdirs = [p for p in outputs_dir.iterdir() if p.is_dir() and p.name != "for_r"]
    if len(subdirs) < n:
        raise FileNotFoundError(f"Fewer than {n} batch folders found in: {outputs_dir}")
    return sorted(subdirs, key=lambda p: p.stat().st_mtime, reverse=True)[:n]

def get_combination(batch_dir):
    combination_file = batch_dir / "combination.txt"
    if not combination_file.exists():
        raise FileNotFoundError(f"No combination.txt found in: {batch_dir}")
    first_line = combination_file.read_text(encoding="utf-8").splitlines()[0]
    return first_line.split("=", 1)[1].strip()


In [4]:
batch_dirs = get_recent_batch_dirs(OUTPUTS_DIR)
results_by_combo = {}
for batch_dir in batch_dirs:
    combination = get_combination(batch_dir)
    print(f"Loading {combination}: {batch_dir}")
    results_list, pickle_paths = load_all_result_pickles(batch_dir)
    print(f"Loaded {len(results_list)} result files")
    results_by_combo[combination] = results_list
    print(f"{combination}: loaded {len(results_list)} result files")

TT_results_list = results_by_combo["TT"]
TF_results_list = results_by_combo["TF"]
FT_results_list = results_by_combo["FT"]
FF_results_list = results_by_combo["FF"]

Loading FF: C:\Users\sarah\Downloads\Emotion-Contagion\BigProject\outputs\2026-09-10_22-31-12
Loaded 1800 result files
FF: loaded 1800 result files
Loading FT: C:\Users\sarah\Downloads\Emotion-Contagion\BigProject\outputs\2026-09-10_22-23-11
Loaded 1800 result files
FT: loaded 1800 result files
Loading TF: C:\Users\sarah\Downloads\Emotion-Contagion\BigProject\outputs\2026-09-10_22-13-06
Loaded 1800 result files
TF: loaded 1800 result files
Loading TT: C:\Users\sarah\Downloads\Emotion-Contagion\BigProject\outputs\2026-09-10_22-02-24
Loaded 1800 result files
TT: loaded 1800 result files


In [5]:
for combination, results_list in results_by_combo.items():
    globals()[f"{combination}_avg_df"] = build_avg_emotion_df(results_list)
    globals()[f"{combination}_intervention_freq_df"] = build_intervention_frequency_df(results_list)
    globals()[f"{combination}_intervention_summary_df"] = build_intervention_summary_df(results_list)
    globals()[f"{combination}_intervention_raster_df"] = build_intervention_raster_df(results_list)
    globals()[f"{combination}_rl_action_df"] = build_rl_action_df(results_list)
    globals()[f"{combination}_rl_quality_df"] = build_rl_quality_df(results_list)
    globals()[f"{combination}_homophily_df"] = build_homophily_df(results_list)

### What the objects look like (give this to the LLM of your choice when asking for analysis help, so they know how results are structured)
@dataclass
class SimulationResults:
    run_id: Any                                  # scalar
    seed: int                                    # scalar

    state: SimulationState                       # final state object
      state.agents: List[Dict[str, Any]]         # [agent]
      state.leader_index: int                    # scalar
      state.intimacy_matrix: np.ndarray          # [agent, agent]
      state.assignments: np.ndarray | None       # [agent]
      state.time: int                            # scalar
      state.metadata: Dict[str, Any]             # config / run metadata

    initial_conditions: pd.DataFrame             # [agent, variable]
    emotion_history: List[List[float]]           # [t_state][agent]; t_state = 0 is the initial state before timestep 0, len(emotion_history) = max_iterations + 1
    avg_emotion_history: List[float]             # [t], member-only avg emotion per timestep, len(avg_emotion_history) = max_iterations
    buddies_per_timestep: List[List[Tuple[int, int]]]   # [t][interaction_pair]; each pair is (i, j) in full-agent indexing, member-member only
    interactions_per_timestep: List[int]         # [t]; number of member-member interactions at each timestep
    intervention_timesteps: List[int]            # [t_intervention]; sparse list of timesteps where leader intervened
    absorption_history: List[Dict[Tuple[int, int], float]]   # [t][(i, j)] cumulative absorption; cumulative absorption/change dictionary after each timestep
    final_absorption_dict: Dict[Tuple[int, int], float]      # [(i, j)] final cumulative absorption
    metadata: Dict[str, Any]                     # run-level metadata / parameters (like: network specifications, leader_style, condition_name, population_size, max_iterations, adaptive_intimacy)
    leader_summary: Dict[str, Any]               # small final leader summary (style, leader_index, leader_emotion, emotionManagementAbility, interventionThreshold, dampening)

batch = {
    "results": List[SimulationResults],          # [run]
    "summary_df": pd.DataFrame | None,           # [run, summary_variable]
    "output_dir": Path,                          # scalar
    "summary_path": Path | None,                 # scalar
    "conditions_used": List[Dict[str, Any]]     # [condition]
}

In [52]:
COMBINATIONS = ["TT", "TF", "FT", "FF"]
COMBINATION_LABELS = {
    "TT": "Adaptive + Leader Ties",
    "TF": "Adaptive + No Leader Ties",
    "FT": "Fixed + Leader Ties",
    "FF": "Fixed + No Leader Ties",
}

NETWORKS = ["random", "community", "core_periphery"]
NETWORK_LABELS = {"random": "Random", "community": "Community", "core_periphery": "Core-periphery"}

LEADERS = ["No_Intervention", "High_Fully_Constrained", "Low_Fully_Constrained", "High_Initially_Constrained", "Low_Initially_Constrained", "Free"]
LEADER_LABELS = {
    "No_Intervention": "No Intervention",
    "High_Fully_Constrained": "High Fully Constrained",
    "Low_Fully_Constrained": "Low Fully Constrained",
    "High_Initially_Constrained": "High Initially Constrained",
    "Low_Initially_Constrained": "Low Initially Constrained",
    "Free": "Free",
}

In [10]:
def build_emotion_sd_df(results_by_combo):
    rows = []
    for combination, results_list in results_by_combo.items():
        for results in results_list:
            structure = results.metadata.get("structure")
            leader_style = results.metadata.get("leader_style")
            avg_history = results.avg_emotion_history
            emotion_history = results.emotion_history
            member_emotion_history = emotion_history[1:]
            for t, avg_emotion in enumerate(avg_history):
                member_emotions = np.asarray(member_emotion_history[t], dtype=float)

                # Leader is always the last element.
                member_emotions = member_emotions[:-1]
                sd = np.std(member_emotions, ddof=0)
                rows.append({
                    "combination": combination,
                    "combination_label": COMBINATION_LABELS.get(combination, combination),
                    "run_id": results.run_id,
                    "seed": results.seed,
                    "structure": structure,
                    "network": NETWORK_LABELS.get(structure, structure),
                    "leader_style": leader_style,
                    "leader": LEADER_LABELS.get(leader_style, leader_style),
                    "time": t,
                    "avg_member_emotion": avg_emotion,
                    "member_emotion_sd": sd,
                })
    return pd.DataFrame(rows)

emotion_sd_df = build_emotion_sd_df(results_by_combo)

print(f"Rows: {len(emotion_sd_df):,}")
print(f"Runs: {emotion_sd_df['run_id'].nunique():,}")
print(f"Combinations: {emotion_sd_df['combination'].unique()}")
print(f"Networks: {emotion_sd_df['structure'].unique()}")
print(f"Leaders: {emotion_sd_df['leader_style'].unique()}")

Rows: 2,160,000
Runs: 1,800
Combinations: ['FF' 'FT' 'TF' 'TT']
Networks: ['random' 'community' 'core_periphery']
Leaders: ['No_Intervention' 'High_Initially_Constrained'
 'Low_Initially_Constrained' 'High_Fully_Constrained' 'Free'
 'Low_Fully_Constrained']


# Data integrity checks

In [6]:
def check_results_consistency(results_list, combination):
    expected_runs = 1800
    expected_repetitions = 100
    expected_population = 10
    expected_total_agents = expected_population + 1
    expected_iterations = 300
    print(f"Checking {combination}")
    print(f"{'=' * 11}")

    # 1. Number of runs
    print(f"Number of runs: {len(results_list)}", end=" ")
    print("✓" if len(results_list) == expected_runs else "WARNING")

    # 2. Condition counts
    condition_counts = (pd.DataFrame([{"structure": r.metadata.get("structure"), "leader_style": r.metadata.get("leader_style")} for r in results_list]).value_counts())
    expected_condition_count = expected_repetitions
    condition_ok = all(count == expected_condition_count for count in condition_counts)
    print(f"Condition counts: {expected_condition_count} repetitions each", "✓" if condition_ok else "WARNING")
    #if not condition_ok: display(condition_counts.reset_index(name="n_runs"))

    # 3. Duplicate run IDs
    run_ids = [r.run_id for r in results_list]
    duplicate_run_ids = len(run_ids) - len(set(run_ids))
    print(f"Duplicate run IDs: {duplicate_run_ids}", "✓" if duplicate_run_ids == 0 else "WARNING")

    # 4. Duplicate seeds within conditions
    seed_df = pd.DataFrame([{"structure": r.metadata.get("structure"), "leader_style": r.metadata.get("leader_style"), "seed": r.seed} for r in results_list])
    duplicate_seeds = seed_df.duplicated(subset=["structure", "leader_style", "seed"]).sum()
    print(f"Duplicate seeds within conditions: {duplicate_seeds}", "✓" if duplicate_seeds == 0 else "WARNING")

    # 5. Time-series lengths
    expected_emotion_length = expected_iterations + 1
    emotion_lengths = {len(r.emotion_history) for r in results_list}
    avg_emotion_lengths = {len(r.avg_emotion_history) for r in results_list}
    interactions_lengths = {len(r.interactions_per_timestep) for r in results_list}
    buddies_lengths = {len(r.buddies_per_timestep) for r in results_list}
    absorption_lengths = {len(r.absorption_history) for r in results_list}
    emotion_ok = emotion_lengths == {expected_emotion_length}
    avg_emotion_ok = avg_emotion_lengths == {expected_iterations}
    interactions_ok = interactions_lengths == {expected_iterations}
    buddies_ok = buddies_lengths == {expected_iterations}
    absorption_ok = absorption_lengths == {expected_iterations}
    print(f"emotion_history length: {emotion_lengths}", "✓" if emotion_ok else "WARNING")
    print(f"avg_emotion_history length: {avg_emotion_lengths}", "✓" if avg_emotion_ok else "WARNING")
    print(f"interactions_per_timestep length: {interactions_lengths}", "✓" if interactions_ok else "WARNING")
    print(f"buddies_per_timestep length: {buddies_lengths}", "✓" if buddies_ok else "WARNING")
    print(f"absorption_history length: {absorption_lengths}", "✓" if absorption_ok else "WARNING")

    # 6. Agent / matrix dimensions
    agent_counts = {len(r.state.agents) for r in results_list}
    matrix_shapes = {r.state.intimacy_matrix.shape for r in results_list}
    agents_ok = agent_counts == {expected_total_agents}
    matrix_ok = matrix_shapes == {(expected_total_agents, expected_total_agents)}
    print(f"Number of agents: {agent_counts}", "✓" if agents_ok else "WARNING")
    print(f"Intimacy matrix shape: {matrix_shapes}", "✓" if matrix_ok else "WARNING")

    # 7. Assignments
    assignment_lengths = {len(r.state.assignments) for r in results_list if r.state.assignments is not None}
    assignments_ok = (not assignment_lengths or assignment_lengths == {expected_total_agents})
    print(f"Assignment length: {assignment_lengths}", "✓" if assignments_ok else "WARNING")

    # 8. DataFrame null checks
    print("\n")
    return {
        "runs": len(results_list),
        "condition_counts": condition_counts,
        "duplicate_run_ids": duplicate_run_ids,
        "duplicate_seeds": duplicate_seeds,
        "emotion_lengths": emotion_lengths,
        "avg_emotion_lengths": avg_emotion_lengths,
        "interactions_lengths": interactions_lengths,
        "buddies_lengths": buddies_lengths,
        "absorption_lengths": absorption_lengths,
        "agent_counts": agent_counts,
        "matrix_shapes": matrix_shapes,
        "assignment_lengths": assignment_lengths,
    }

TT_check = check_results_consistency(TT_results_list, "TT")
TF_check = check_results_consistency(TF_results_list, "TF")
FT_check = check_results_consistency(FT_results_list, "FT")
FF_check = check_results_consistency(FF_results_list, "FF")

Checking TT
Number of runs: 1800 ✓
Condition counts: 100 repetitions each ✓
Duplicate run IDs: 0 ✓
Duplicate seeds within conditions: 0 ✓
emotion_history length: {301} ✓
avg_emotion_history length: {300} ✓
interactions_per_timestep length: {300} ✓
buddies_per_timestep length: {300} ✓
absorption_history length: {300} ✓
Number of agents: {10} WARNING
Intimacy matrix shape: {(10, 10)} WARNING
Assignment length: {10} WARNING


Checking TF
Number of runs: 1800 ✓
Condition counts: 100 repetitions each ✓
Duplicate run IDs: 0 ✓
Duplicate seeds within conditions: 0 ✓
emotion_history length: {301} ✓
avg_emotion_history length: {300} ✓
interactions_per_timestep length: {300} ✓
buddies_per_timestep length: {300} ✓
absorption_history length: {300} ✓
Number of agents: {10} WARNING
Intimacy matrix shape: {(9, 9)} WARNING
Assignment length: {9} WARNING


Checking FT
Number of runs: 1800 ✓
Condition counts: 100 repetitions each ✓
Duplicate run IDs: 0 ✓
Duplicate seeds within conditions: 0 ✓
emotion_his

In [7]:
def check_dataframe_nulls(df, name):
    null_count = df.isnull().sum().sum()
    print(f"{name}: {null_count} null values", "✓" if null_count == 0 else "WARNING")

check_dataframe_nulls(TT_avg_df, "TT_avg_df")
check_dataframe_nulls(TT_intervention_freq_df, "TT_intervention_freq_df")
check_dataframe_nulls(TT_intervention_summary_df, "TT_intervention_summary_df")
check_dataframe_nulls(TT_intervention_raster_df, "TT_intervention_raster_df")
check_dataframe_nulls(TT_rl_action_df, "TT_rl_action_df")
check_dataframe_nulls(TT_rl_quality_df, "TT_rl_quality_df")
check_dataframe_nulls(TT_homophily_df, "TT_homophily_df")

TT_avg_df: 540000 null values WARNING
TT_intervention_freq_df: 540000 null values WARNING
TT_intervention_summary_df: 7800 null values WARNING
TT_intervention_raster_df: 81193 null values WARNING
TT_rl_action_df: 540000 null values WARNING
TT_rl_quality_df: 451500 null values WARNING
TT_homophily_df: 541800 null values WARNING


In [18]:
FT_avg_df.isnull().sum()

run_id                     0
seed                       0
condition_name        540000
structure                  0
leader_style               0
time                       0
avg_member_emotion         0
dtype: int64

In [39]:
duplicates = emotion_sd_df.duplicated(subset=["combination", "run_id", "structure", "leader_style", "time"])
print("Duplicate rows:", duplicates.sum())

Duplicate rows: 0


In [11]:
print("Missing average emotion:")
print(emotion_sd_df["avg_member_emotion"].isna().sum())

print("\nMissing SD:")
print(emotion_sd_df["member_emotion_sd"].isna().sum())

print("\nRows per combination:")
print(emotion_sd_df.groupby("combination")["run_id"].nunique())

print("\nRows per network:")
print(emotion_sd_df.groupby("structure")["run_id"].nunique())

print("\nRows per leader:")
print(emotion_sd_df.groupby("leader_style")["run_id"].nunique())

Missing average emotion:
0

Missing SD:
0

Rows per combination:
combination
FF    1800
FT    1800
TF    1800
TT    1800
Name: run_id, dtype: int64

Rows per network:
structure
community         600
core_periphery    600
random            600
Name: run_id, dtype: int64

Rows per leader:
leader_style
Free                          300
High_Fully_Constrained        300
High_Initially_Constrained    300
Low_Fully_Constrained         300
Low_Initially_Constrained     300
No_Intervention               300
Name: run_id, dtype: int64


In [12]:
print("Missing average emotion:")
print(emotion_sd_df["avg_member_emotion"].isna().sum())

print("\nMissing SD:")
print(emotion_sd_df["member_emotion_sd"].isna().sum())

print("\nRows per combination:")
print(emotion_sd_df.groupby("combination")["run_id"].nunique())

print("\nRows per network:")
print(emotion_sd_df.groupby("structure")["run_id"].nunique())

print("\nRows per leader:")
print(emotion_sd_df.groupby("leader_style")["run_id"].nunique())

Missing average emotion:
0

Missing SD:
0

Rows per combination:
combination
FF    1800
FT    1800
TF    1800
TT    1800
Name: run_id, dtype: int64

Rows per network:
structure
community         600
core_periphery    600
random            600
Name: run_id, dtype: int64

Rows per leader:
leader_style
Free                          300
High_Fully_Constrained        300
High_Initially_Constrained    300
Low_Fully_Constrained         300
Low_Initially_Constrained     300
No_Intervention               300
Name: run_id, dtype: int64


## Interactive dashboard

In [ ]:
# Combinations
combination_options = [("Adaptive + Leader Ties", "TT"),  ("Adaptive + No Leader Ties", "TF"), ("Fixed + Leader Ties", "FT"), ("Fixed + No Leader Ties", "FF")]
combination_select = widgets.SelectMultiple(options=combination_options, value=("TT", "TF", "FT", "FF"), description="Conditions:", rows=4, layout=widgets.Layout(width="350px"))

# Network selection
network_options = [("Random", "random"), ("Community", "community"), ("Core-periphery", "core_periphery")]
network_select = widgets.SelectMultiple(options=network_options, value=("random", "community", "core_periphery"), description="Networks:", rows=3, layout=widgets.Layout(width="300px"))

# Leader selection
leader_options = [
    ("No Intervention", "No_Intervention"),
    ("High Fully Constrained", "High_Fully_Constrained"),
    ("Low Fully Constrained", "Low_Fully_Constrained"),
    ("High Initially Constrained", "High_Initially_Constrained"),
    ("Low Initially Constrained", "Low_Initially_Constrained"),
    ("Free", "Free"),
]
leader_select = widgets.SelectMultiple(options=leader_options, value=tuple(LEADERS), description="Leaders:", rows=6, layout=widgets.Layout(width="350px"))

# SD threshold
first_sd_slider = widgets.FloatSlider(value=0.5, min=0.0, max=3.0, step=0.01, description="SD ≤", continuous_update=False, readout_format=".2f", layout=widgets.Layout(width="400px"))
first_sd_text = widgets.FloatText(value=0.5, description="SD value:", step=0.01, layout=widgets.Layout(width="180px"))

# Keep slider and text box synchronized
def update_first_sd_text(change):
    if change["name"] == "value": first_sd_text.value = change["new"]

def update_first_sd_slider(change):
    if change["name"] == "value":
        value = change["new"]
        if value < first_sd_slider.min: value = first_sd_slider.min
        if value > first_sd_slider.max: value = first_sd_slider.max
        if value != first_sd_slider.value: first_sd_slider.value = value

first_sd_slider.observe(update_first_sd_text, names="value")
first_sd_text.observe(update_first_sd_slider, names="value")

# Optional average-emotion criterion
emotion_threshold_checkbox = widgets.Checkbox( value=False, description="Require average emotion ≥ threshold")
emotion_threshold = widgets.FloatText(value=0.5, description="Emotion ≥", step=0.05, disabled=True, layout=widgets.Layout(width="200px"))

def toggle_emotion_threshold(change):  emotion_threshold.disabled = not change["new"]

emotion_threshold_checkbox.observe(toggle_emotion_threshold, names="value")

In [78]:
def get_filtered_data(df, combinations, networks, leaders):
    filtered = df[df["combination"].isin(combinations) & df["structure"].isin(networks) & df["leader_style"].isin(leaders)].copy()
    return filtered

def calculate_first_convergence(df, sd_threshold, use_emotion_threshold=False, emotion_threshold_value=0.0):
    """
    Calculate the first timestep at which each simulation
    satisfies the selected convergence criteria.
    """
    if df.empty: 
        return pd.DataFrame()
    
    converged = df[df["member_emotion_sd"] <= sd_threshold].copy()
    if use_emotion_threshold:
        converged = converged[converged["avg_member_emotion"] >= emotion_threshold_value]
    first_convergence = (converged.groupby(["combination", "run_id", "seed", "structure", "leader_style"], as_index=False)["time"].min().rename(columns={"time": "first_convergence"}))

    # Add simulations that never converged.
    all_runs = (df[["combination", "run_id", "seed", "structure", "leader_style"]].drop_duplicates())
    first_convergence = all_runs.merge(first_convergence, on=["combination", "run_id", "seed", "structure", "leader_style"], how="left")

    return first_convergence

def format_convergence_value(value):
    if pd.isna(value):
        return "--"
    return f"{value:.1f}"

def make_average_emotion_plot(df):
    fig = go.Figure()
    if df.empty:
        fig.update_layout(title="Average Member Emotion", xaxis_title="Timestep", yaxis_title="Average Member Emotion")
        return fig
    grouped = (df.groupby(["combination", "structure", "leader_style", "time"], as_index=False).agg(mean_emotion=("avg_member_emotion", "mean")))
    for (combination, structure, leader_style), group in grouped.groupby(["combination", "structure", "leader_style"]):
        fig.add_trace(
            go.Scatter(
                x=group["time"],
                y=group["mean_emotion"],
                mode="lines",
                name=(
                    f"{COMBINATION_LABELS.get(combination, combination)} | "
                    f"{NETWORK_LABELS.get(structure, structure)} | "
                    f"{LEADER_LABELS.get(leader_style, leader_style)}"
                )
            )
        )
    fig.update_layout(title="Average Member Emotion Over Time", xaxis_title="Timestep", yaxis_title="Average Member Emotion", hovermode="x unified", height=600, legend=dict(orientation="v"))

    return fig

def make_sd_plot(df, sd_threshold):
    fig = go.Figure()
    if not df.empty:
        grouped = (df.groupby(["combination", "structure", "leader_style", "time"], as_index=False).agg(mean_sd=("member_emotion_sd", "mean")))
        for (combination, structure, leader_style), group in grouped.groupby(["combination", "structure", "leader_style"]):
            fig.add_trace(
                go.Scatter(
                    x=group["time"],
                    y=group["mean_sd"],
                    mode="lines",
                    name=(
                        f"{COMBINATION_LABELS.get(combination, combination)} | "
                        f"{NETWORK_LABELS.get(structure, structure)} | "
                        f"{LEADER_LABELS.get(leader_style, leader_style)}"
                    )
                )
            )

    fig.add_hline(y=sd_threshold, line_dash="dash", annotation_text=f"SD threshold = {sd_threshold:.2f}")
    fig.update_layout(title="Average Member Emotion SD Over Time", xaxis_title="Timestep", yaxis_title="Member Emotion SD", hovermode="x unified", height=600)

    return fig

def make_network_leader_table(convergence_df):
    if convergence_df.empty: 
        return pd.DataFrame()

    table = (convergence_df.groupby(["leader_style", "structure"], as_index=False).agg(first_convergence=("first_convergence", "mean")))
    table["leader"] = table["leader_style"].map(LEADER_LABELS)
    table["network"] = table["structure"].map(NETWORK_LABELS)
    table = table.pivot(index="leader", columns="network", values="first_convergence")

    # Keep network order
    desired_columns = [NETWORK_LABELS[n] for n in NETWORKS if NETWORK_LABELS[n] in table.columns]
    table = table.reindex(columns=desired_columns)
    table = table.reindex([LEADER_LABELS[l] for l in LEADERS if LEADER_LABELS[l] in table.index])

    table = table.round(1)
    table = table.fillna("--")  # for prettiness

    return table

def make_network_summary(convergence_df):
    if convergence_df.empty:
        return pd.DataFrame()

    summary = (convergence_df.groupby("structure").agg(mean_first_convergence=( "first_convergence", "mean"), convergence_rate=("first_convergence", lambda x: x.notna().mean()), simulations=("run_id", "count")).reset_index())
    summary["network"] = summary["structure"].map(NETWORK_LABELS)
    summary["convergence_rate"] = (summary["convergence_rate"] * 100).round(1)
    summary["mean_first_convergence"] = (summary["mean_first_convergence"].round(1))
    summary = summary[["network", "mean_first_convergence", "convergence_rate", "simulations"]]
    summary = summary.rename(
        columns={
            "mean_first_convergence": "Mean First Convergence",
            "convergence_rate": "Convergence Rate (%)",
            "simulations": "Simulations"
        }
    )
    summary = summary.fillna("--")

    return summary

def make_leader_summary(convergence_df):
    if convergence_df.empty: return pd.DataFrame()

    summary = (convergence_df.groupby("leader_style").agg(mean_first_convergence=("first_convergence","mean"), convergence_rate=("first_convergence",lambda x: x.notna().mean()), simulations=("run_id", "count")).reset_index())
    summary["leader"] = summary["leader_style"].map(LEADER_LABELS)
    summary["convergence_rate"] = (summary["convergence_rate"] * 100).round(1)
    summary["mean_first_convergence"] = (summary["mean_first_convergence"].round(1))
    summary = summary[["leader", "mean_first_convergence", "convergence_rate", "simulations"]]
    summary = summary.rename(
        columns={
            "mean_first_convergence": "Mean First Convergence",
            "convergence_rate": "Convergence Rate (%)",
            "simulations": "Simulations"
        }
    )
    summary = summary.fillna("--")
    return summary

In [87]:
first_dashboard_output = widgets.Output()
def update_dashboard(combinations, networks, leaders, sd_threshold, emotion_threshold_enabled,  emotion_threshold_value):
    with first_dashboard_output:
        clear_output(wait=True)
        combinations = list(combinations)
        networks = list(networks)
        leaders = list(leaders)

        # Filter data
        filtered_df = get_filtered_data(emotion_sd_df, combinations, networks, leaders)
        if filtered_df.empty:
            display(widgets.HTML("<b>No data match the current selections.</b>"))
            return
        
        # Calculate convergence
        convergence_df = calculate_first_convergence(filtered_df, sd_threshold, use_emotion_threshold=emotion_threshold_enabled, emotion_threshold_value=emotion_threshold_value)

        # Display current settings
        criterion_text = (f"SD ≤ {sd_threshold:.2f}")
        if emotion_threshold_enabled:
            criterion_text += (f" AND average emotion ≥ " f"{emotion_threshold_value:.2f}")

        display(
            widgets.HTML(
                f"""
                <h3>Current Selection</h3>
                <p>
                <b>Conditions:</b>
                {", ".join(COMBINATION_LABELS.get(c, c) for c in combinations)}
                </p>
                <p>
                <b>Networks:</b>
                {", ".join(NETWORK_LABELS.get(n, n) for n in networks)}
                </p>
                <p>
                <b>Leaders:</b>
                {", ".join(LEADER_LABELS.get(l, l) for l in leaders)}
                </p>
                <p>
                <b>Convergence criteria:</b>
                {criterion_text}
                </p>
                """
            )
        )

        # Plots
        display(make_average_emotion_plot(filtered_df))
        display(make_sd_plot(filtered_df, sd_threshold))

        # Convergence table
        display(widgets.HTML("<h3>First Convergence Timestep: Network × Leader</h3>"))

        network_leader_table = make_network_leader_table(convergence_df)

        if network_leader_table.empty:
            display(widgets.HTML("No simulations reached convergence."))
        else:
            display(network_leader_table)


        # Network summary
        display( widgets.HTML("<h3>First Convergence by Network</h3>"))
        network_summary = make_network_summary(convergence_df)
        display(network_summary.style.format({"Mean First Convergence": "{:.1f}", "Convergence Rate (%)": "{:.1f}"}).hide(axis="index"))

        # Leader summary
        display(widgets.HTML("<h3>First Convergence by Leader</h3>"))
        leader_summary = make_leader_summary(convergence_df)
        display(leader_summary.style.format({"Mean First Convergence": "{:.1f}", "Convergence Rate (%)": "{:.1f}"}).hide(axis="index"))

first_dashboard_controls = widgets.VBox([
    widgets.HTML("<h3>Dashboard Controls</h3>"),
    widgets.HBox([combination_select, network_select, leader_select]),
    widgets.HBox([first_sd_slider, first_sd_text]),
    widgets.HBox([emotion_threshold_checkbox, emotion_threshold])
    ])

def first_dashboard_changed(change):
    update_dashboard(
        combinations=combination_select.value,
        networks=network_select.value,
        leaders=leader_select.value,
        sd_threshold=first_sd_slider.value,
        emotion_threshold_enabled=emotion_threshold_checkbox.value,
        emotion_threshold_value=emotion_threshold.value
    )

combination_select.observe(first_dashboard_changed, names="value")
network_select.observe(first_dashboard_changed, names="value")
leader_select.observe(first_dashboard_changed, names="value")
first_sd_slider.observe(first_dashboard_changed, names="value")
first_sd_text.observe(first_dashboard_changed, names="value")
emotion_threshold_checkbox.observe(first_dashboard_changed, names="value")
emotion_threshold.observe(first_dashboard_changed, names="value")

display(first_dashboard_controls)
display(first_dashboard_output)

# dashboard display
update_dashboard(
    combinations=combination_select.value,
    networks=network_select.value,
    leaders=leader_select.value,
    sd_threshold=first_sd_slider.value,
    emotion_threshold_enabled=emotion_threshold_checkbox.value,
    emotion_threshold_value=emotion_threshold.value
)

Output()

In [ ]:
# Because there are some overlapping lines, i wanted to check the max difference between a sample pair of groups and it's small, but they aren't identical
group_a = (emotion_sd_df[(emotion_sd_df["combination"] == "TT") & (emotion_sd_df["structure"] == "random") & (emotion_sd_df["leader_style"] == "Free")].groupby("time")["avg_member_emotion"].mean())
group_b = (emotion_sd_df[(emotion_sd_df["combination"] == "TF") & (emotion_sd_df["structure"] == "random") & (emotion_sd_df["leader_style"] == "Free")].groupby("time")["avg_member_emotion"].mean())
difference = (group_a - group_b).abs()

print("Maximum difference:", difference.max())
print("Number of different timesteps:", (difference > 1e-12).sum())

Maximum difference: 0.021814367085624453
Number of different timesteps: 300


## Statistical Analysis
#### <u>Focus</u>: **convergence**

### Descriptive

In [88]:
def make_convergence_rate_table(convergence_df, combination):
    # select one implementation condition
    data = convergence_df[convergence_df["combination"] == combination].copy()

    if data.empty:
        return pd.DataFrame()

    # calculate the proportion of simulations that converged
    summary = (data.groupby(["leader_style", "structure"])["first_convergence"].apply(lambda x: x.notna().mean() * 100).reset_index(name="convergence_rate"))

    # reshape into leader rows and network columns
    table = summary.pivot(index="leader_style", columns="structure", values="convergence_rate")

    # apply display labels
    table.index = [LEADER_LABELS.get(x, x) for x in table.index]
    table.columns = [NETWORK_LABELS.get(x, x) for x in table.columns]

    # preserve desired ordering
    desired_rows = [LEADER_LABELS[x] for x in LEADERS if LEADER_LABELS[x] in table.index]
    desired_columns = [NETWORK_LABELS[x] for x in NETWORKS if NETWORK_LABELS[x] in table.columns]

    table = table.reindex(index=desired_rows, columns=desired_columns)

    # format percentages
    table = table.round(1)

    return table

def make_speed_summary(convergence_df):
    # keep simulations that reached convergence
    data = convergence_df[convergence_df["first_convergence"].notna()].copy()

    if data.empty:
        return pd.DataFrame()

    # calculate distribution statistics
    summary = (data.groupby(["combination", "structure", "leader_style"])["first_convergence"]
        .agg(
            converged="count",
            mean="mean",
            median="median",
            sd="std",
            q1=lambda x: x.quantile(0.25),
            q3=lambda x: x.quantile(0.75),
            minimum="min",
            maximum="max"
        ).reset_index())

    # calculate convergence rate using all simulations
    total = (convergence_df.groupby(["combination", "structure", "leader_style"]).size().reset_index(name="total"))

    summary = summary.merge(total, on=["combination", "structure", "leader_style"], how="right")
    summary["convergence_rate"] = (summary["converged"] / summary["total"] * 100)

    # apply display labels
    summary["condition"] = summary["combination"].map(COMBINATION_LABELS)
    summary["network"] = summary["structure"].map(NETWORK_LABELS)
    summary["leader"] = summary["leader_style"].map(LEADER_LABELS)

    # select and order columns
    summary = summary[["combination", "condition", "structure", "network", "leader_style", "leader", "converged", "total", "convergence_rate", "mean", "median", "sd", "q1", "q3", "minimum", "maximum"]]

    return summary

PLOT_COLORS = [
    "#1f77b4",
    "#d62728",
    "#2ca02c",
    "#9467bd",
    "#ff7f0e",
    "#17becf",
]

def make_raincloud_plot(convergence_df, combination):
    # keep one implementation condition
    data = convergence_df[ convergence_df["combination"] == combination].copy()
    fig = go.Figure()

    for network_index, network in enumerate(NETWORKS):
        network_data = data[data["structure"] == network]

        for leader_index, leader in enumerate(LEADERS):
            values = network_data[network_data["leader_style"] == leader]["first_convergence"].dropna()

            if values.empty:
                continue

            # place each network in its own x-axis position
            x_position = network_index
            color = PLOT_COLORS[leader_index]

            # add distribution
            fig.add_trace(go.Violin(
                    x=np.full(len(values), x_position),
                    y=values,
                    name=LEADER_LABELS[leader],
                    legendgroup=leader,
                    scalegroup=leader,
                    side="positive",
                    width=0.12,
                    points="all",
                    jitter=0.25,
                    line=dict(color=color),
                    fillcolor=color,
                    opacity=0.45,
                    showlegend=(network_index == 0),
                    hovertemplate=(
                        f"{NETWORK_LABELS[network]}<br>"
                        f"{LEADER_LABELS[leader]}<br>"
                        "First convergence: %{y}<extra></extra>"
                    )))

            # add box summary
            fig.add_trace(go.Box(
                    x=np.full(len(values), x_position),
                    y=values,
                    name=LEADER_LABELS[leader],
                    legendgroup=leader,
                    boxpoints=False,
                    width=0.08,
                    line=dict(color=color),
                    fillcolor="rgba(255,255,255,0)",
                    showlegend=False,
                    hovertemplate=(
                        f"{NETWORK_LABELS[network]}<br>"
                        f"{LEADER_LABELS[leader]}<br>"
                        "First convergence: %{y}<extra></extra>"
                    )))

    fig.update_layout(
        title=(
            f"First Convergence Time Distribution — "
            f"{COMBINATION_LABELS[combination]}"
        ), 
        xaxis=dict(tickmode="array", tickvals=list(range(len(NETWORKS))), ticktext=[NETWORK_LABELS[x] for x in NETWORKS], title="Network Structure"), yaxis=dict(title="First Convergence Timestep"), height=700, violinmode="overlay", boxmode="overlay", hovermode="closest", legend_title="Leader Style")

    return fig

def make_speed_table(speed_summary, combination):
    # select one implementation condition
    table = speed_summary[speed_summary["combination"] == combination].copy()
    if table.empty:
        return pd.DataFrame()

    table = table[["network", "leader", "converged", "total", "convergence_rate", "mean", "median", "sd", "q1", "q3", "minimum", "maximum"]].copy()
    table = table.rename(columns={
            "network": "Network",
            "leader": "Leader",
            "converged": "Converged",
            "total": "Total",
            "convergence_rate": "Convergence Rate (%)",
            "mean": "Mean",
            "median": "Median",
            "sd": "SD",
            "q1": "Q1",
            "q3": "Q3",
            "minimum": "Min",
            "maximum": "Max",
    })

    table["Convergence Rate (%)"] = (table["Convergence Rate (%)"].round(1))
    numeric_columns = ["Mean", "Median", "SD", "Q1", "Q3", "Min", "Max"]
    table[numeric_columns] = (table[numeric_columns].round(1))

    return table

condition_select = widgets.Dropdown(options=[(COMBINATION_LABELS[x], x) for x in COMBINATIONS], value="TT", description="Condition:", layout=widgets.Layout(width="350px"))
sd_slider2 = widgets.FloatSlider(value=0.01, min=0.0, max=3.0, step=0.001, description="SD ≤", continuous_update=False, readout_format=".3f", layout=widgets.Layout(width="450px"))
sd_text2 = widgets.FloatText(value=0.01,description="SD value:", step=0.001, layout=widgets.Layout(width="200px"))

def update_second_sd_text(change):
    if change["name"] == "value":
        sd_text2.value = change["new"]

def update_second_sd_slider(change):
    if change["name"] == "value":
        value = change["new"]
        if value < sd_slider2.min:
            value = sd_slider2.min

        if value > sd_slider2.max:
            value = sd_slider2.max

        if value != sd_slider2.value:
            sd_slider2.value = value

sd_slider2.observe(update_second_sd_text, names="value")
sd_text2.observe(update_second_sd_slider, names="value")

In [89]:
second_dashboard_output = widgets.Output()
def update_convergence_dashboard(combination, sd_threshold):
    with second_dashboard_output:
        clear_output(wait=True)

        # calculate convergence using current threshold
        convergence_df = calculate_first_convergence(emotion_sd_df, sd_threshold=sd_threshold)

        # calculate numerical summaries
        speed_summary = make_speed_summary(convergence_df)

        # display threshold
        display(widgets.HTML(
            f"""
            <h3>Convergence Analysis</h3>
            <p><b>SD convergence threshold: </b>{sd_threshold:.3f}</p>
            """
        ))

        # display convergence rates
        display(widgets.HTML("<h3>Convergence Rate by Condition</h3>"))

        # display the four tables individually
        rate_output = widgets.HBox([widgets.Output(layout=widgets.Layout( width="25%")) for _ in COMBINATIONS])
        display(rate_output)

        for output_widget, condition in zip(rate_output.children, COMBINATIONS):

            with output_widget:
                display(widgets.HTML(f"<b>{COMBINATION_LABELS[condition]}</b>"))
                table = make_convergence_rate_table(convergence_df, condition)
                if table.empty:
                    display(pd.DataFrame({"Result": ["No data"]}))
                else:
                    display(table.style.format("{:.1f}%"))


        # display selected condition distribution
        display(make_raincloud_plot(convergence_df, combination))

        # display numerical summary
        display(widgets.HTML("<h3>First Convergence Time Summary</h3>"))
        speed_table = make_speed_table(speed_summary, combination)
        if speed_table.empty:
            display(widgets.HTML("No simulations reached convergence."))
        else:
            display(speed_table.style.format({
                "Convergence Rate (%)": "{:.1f}",
                "Mean": "{:.1f}",
                "Median": "{:.1f}",
                "SD": "{:.1f}",
                "Q1": "{:.1f}",
                "Q3": "{:.1f}",
                "Min": "{:.1f}",
                "Max": "{:.1f}",
            }))

def dashboard_changed(change):
    update_convergence_dashboard(combination=condition_select.value, sd_threshold=sd_slider2.value)

condition_select.observe(dashboard_changed, names="value")
sd_slider2.observe(dashboard_changed, names="value")
sd_text2.observe(dashboard_changed, names="value")

dashboard_controls = widgets.VBox([widgets.HTML("<h3>Controls</h3>"), condition_select, widgets.HBox([sd_slider2, sd_text2])])

display(dashboard_controls)
display(second_dashboard_output)

# display the initial dashboard
update_convergence_dashboard(combination=condition_select.value, sd_threshold=sd_slider2.value)

Output()